# Experiment 6 — Margin-normalization ablation

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** Evaluate how much margin-normalization matters: fit the same
model on (a) raw-dollar prices, (b) only market-normalized prices (divided
by $M$), and (c) the full margin-normalization $(S-v)/(M-v)$. Use
leave-one-product-out CV to expose generalization failures.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

def build_variant(pol_turns, scheme):
    df = pol_turns.copy()
    df = df.sort_values(['run_id', 'round'])
    g = df.groupby('run_id', sort=False)
    # lag 1 + current buyer offer, always
    if scheme == 'raw':
        df['B_use'] = df['B']
        df['S_lag_use'] = g['S'].shift(1)
        df['B_lag_use'] = g['B'].shift(1)
        df['y_use']     = df['S']
    elif scheme == 'market':
        df['B_use']  = df['B'] / df['M']
        df['S_lag_use'] = g['S'].shift(1) / df['M']
        df['B_lag_use'] = g['B'].shift(1) / df['M']
        df['y_use']  = df['S'] / df['M']
    else:  # 'margin' (full v7 normalization)
        df['B_use']  = df['B_norm']
        df['S_lag_use'] = df['S_lag1']
        df['B_lag_use'] = df['B_lag1']
        df['y_use']  = df['S_norm']
    keep = df[['B_use','S_lag_use','B_lag_use','y_use','car_name','vehicle','round']].dropna()
    return keep

def loo_product_mae(policy, scheme):
    pol_turns = turns[(turns['policy'] == policy)
                      & (turns['round'].between(nu.MODEL_ROUND_MIN, nu.MODEL_ROUND_MAX))
                      & (~turns['parse_fail'])]
    data = build_variant(pol_turns, scheme)
    if data.empty:
        return np.nan
    cars = sorted(data['car_name'].unique())
    fold_maes = []
    for held in cars:
        tr = data[data['car_name'] != held]
        te = data[data['car_name'] == held]
        if len(tr) < 50 or len(te) < 20:
            continue
        X_tr = tr[['B_use','S_lag_use','B_lag_use']].to_numpy()
        X_te = te[['B_use','S_lag_use','B_lag_use']].to_numpy()
        y_tr = tr['y_use'].to_numpy()
        y_te = te['y_use'].to_numpy()
        scaler = StandardScaler().fit(X_tr)
        reg = Ridge(alpha=1.0).fit(scaler.transform(X_tr), y_tr)
        yhat = reg.predict(scaler.transform(X_te))
        fold_maes.append(mean_absolute_error(y_te, yhat))
    return float(np.mean(fold_maes)) if fold_maes else np.nan

rows = []
for p in POLICIES:
    for scheme in ['raw', 'market', 'margin']:
        rows.append({'policy': p, 'scheme': scheme,
                     'loo_mae': loo_product_mae(p, scheme)})
ablation = pd.DataFrame(rows).pivot(index='policy', columns='scheme', values='loo_mae')
print('Leave-one-product-out MAE (lower is better):')
print(ablation.round(4))